# 03 · Inference: packaged model and prediction API

**Assignment tasks covered:** Task 4 (using the packaged model) and Task 6 (the containerised API).

Two ways to get a prediction from the trained model:

1. **In Python** – load `models/heart_disease_model.joblib` with `src.predict` (what the API does internally).
2. **Over HTTP** – call the FastAPI service running in Docker (`POST /predict`), exactly as a client application would.

The second part needs the container running:
`docker run -d -p 8000:8000 --name heart-api heart-disease-api:1.0.0`. If it is not running, those cells say so instead of failing.

In [1]:
# Project setup: find the project root so every path matches the scripts in src/
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists():   # e.g. notebooks/ -> project root
    if ROOT == ROOT.parent:
        raise RuntimeError("Open this notebook from inside the project folder.")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT} | Python {sys.version.split()[0]}")

Project root: c:\Users\Snehanjali\heart-disease-prediction-mlops | Python 3.13.5


## 1. The packaged model in Python

In [2]:
import json
import pandas as pd
from src import config
from src.predict import load_model, predict

model = load_model.__wrapped__(str(config.MODEL_PATH))       # fresh load from disk
meta = json.loads(config.MODEL_METADATA_PATH.read_text())
print("Model:", meta["model_name"], "| trained:", meta["trained_at_utc"], "| scikit-learn", meta["sklearn_version"])
print("Pipeline steps:", [name for name, _ in model.steps])

patients = {name: json.loads((ROOT / "samples" / f"{name}.json").read_text())
            for name in ["high_risk", "low_risk", "missing_values"]}
results = predict(list(patients.values()), model=model)
pd.DataFrame(results, index=list(patients)).join(pd.DataFrame(patients).T[["age", "sex", "cp", "thalach", "oldpeak", "ca", "thal"]])

Model: logistic_regression | trained: 2026-10-18T11:43:39+00:00 | scikit-learn 1.9.1
Pipeline steps: ['preprocess', 'model']


,prediction,label,probability_disease,confidence,age,sex,cp,thalach,oldpeak,ca,thal
high_risk,1,Heart disease,0.9787,0.9787,67.0,1.0,4.0,108.0,1.5,3.0,3.0
low_risk,0,No heart disease,0.0841,0.9159,41.0,0.0,2.0,172.0,1.4,0.0,3.0
missing_values,0,No heart disease,0.1181,0.8819,58.0,1.0,3.0,165.0,0.0,NaN,NaN


The **missing_values** patient has `ca` and `thal` set to null: the imputers inside the saved pipeline fill them in,
exactly as during training, so no special handling is needed by the caller.

### Effect of the decision threshold
The model outputs a probability; the label depends on the threshold (0.5 by default, configurable in the API via
`PREDICTION_THRESHOLD`).

In [3]:
for t in (0.3, 0.5, 0.7):
    labels = [r["prediction"] for r in predict(list(patients.values()), model=model, threshold=t)]
    print(f"threshold {t}: predictions {dict(zip(patients, labels))}")

threshold 0.3: predictions {'high_risk': 1, 'low_risk': 0, 'missing_values': 0}
threshold 0.5: predictions {'high_risk': 1, 'low_risk': 0, 'missing_values': 0}
threshold 0.7: predictions {'high_risk': 1, 'low_risk': 0, 'missing_values': 0}


## 2. The same predictions through the API (Docker container)

In [4]:
import urllib.request, urllib.error

API = os.environ.get("API_URL", "http://localhost:8000")

def call(path, body=None):
    req = urllib.request.Request(API + path, method="POST" if body is not None else "GET",
                                 data=json.dumps(body).encode() if body is not None else None,
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=5) as r:
            return r.status, json.loads(r.read())
    except urllib.error.HTTPError as e:
        return e.code, json.loads(e.read())

try:
    status, body = call("/ready")
    API_UP = status == 200
    print("API ready:", body)
except OSError as exc:
    API_UP = False
    print(f"API not reachable at {API} ({exc.__class__.__name__}). Start it with:\n"
          "  docker run -d -p 8000:8000 --name heart-api heart-disease-api:1.0.0")

API ready: {'status': 'ready', 'model_loaded': True, 'model_version': 'logistic_regression-2026-10-18'}


In [5]:
if API_UP:
    rows = []
    for name, patient in patients.items():
        status, body = call("/predict", patient)
        rows.append({"patient": name, "http": status, **{k: body[k] for k in
                     ("prediction", "label", "probability_disease", "confidence", "model_version")}})
    api_df = pd.DataFrame(rows).set_index("patient")
    display(api_df)
    same = (api_df["probability_disease"].values == [r["probability_disease"] for r in results]).all()
    print("API and in-process predictions identical:", bool(same))

,http,prediction,label,probability_disease,confidence,model_version
patient,,,,,,
high_risk,200,1,Heart disease,0.9787,0.9787,logistic_regression-2026-10-18
low_risk,200,0,No heart disease,0.0841,0.9159,logistic_regression-2026-10-18
missing_values,200,0,No heart disease,0.1181,0.8819,logistic_regression-2026-10-18


API and in-process predictions identical: True


### Invalid input is rejected with a clear message (HTTP 422)

In [6]:
if API_UP:
    status, body = call("/predict", json.loads((ROOT / "samples" / "invalid.json").read_text()))
    print("HTTP", status)
    display(pd.DataFrame(body["detail"]))

HTTP 422


,field,message,type
0,age,Input should be less than or equal to 120,less_than_equal
1,cp,"Input should be 1, 2, 3 or 4",literal_error
2,thal,"Input should be 3, 6 or 7",literal_error


### Model information served by the API

In [7]:
if API_UP:
    status, info = call("/model-info")
    print(json.dumps({k: info[k] for k in ("model_version", "model_class", "best_params",
                                            "cv_roc_auc_mean", "test_metrics", "decision_threshold")}, indent=2))

{
  "model_version": "logistic_regression-2026-10-18",
  "model_class": "LogisticRegression",
  "best_params": {
    "C": 0.1,
    "class_weight": null,
    "l1_ratio": 0.0
  },
  "cv_roc_auc_mean": 0.9113,
  "test_metrics": {
    "test_accuracy": 0.8852,
    "test_precision": 0.8387,
    "test_recall": 0.9286,
    "test_f1": 0.8814,
    "test_roc_auc": 0.961
  },
  "decision_threshold": 0.5
}


## Summary

* The packaged pipeline (preprocessing + model in one file) predicts directly from raw clinical values,
  including records with missing `ca` / `thal`.
* The Dockerised API returns **identical** probabilities, adds a confidence score, a model version and a request id,
  and rejects invalid input with per-field messages.